In [0]:
#display data from general_ledgers table in plworkforce_catalog.001_bronze.general_ledgers
display(spark.read.table("plworkforce_catalog.001_bronze.general_ledgers"))

from pyspark.sql import functions as F

# Use silver schema
spark.sql("USE plworkforce_catalog.002_silver")

# Read from Bronze
df = spark.table("plworkforce_catalog.001_bronze.general_ledgers")

# Transform
df_clean = (df

    # Rename columns (standard naming)
    .withColumnRenamed("gl_id", "transaction_id")

    # Convert string → proper date
    .withColumn("entry_date",
        F.to_timestamp("entry_date", "dd-MM-yyyy HH:mm"))

    .withColumn("posting_date",
        F.to_timestamp("posting_date", "dd-MM-yyyy HH:mm"))
    
    # generating amount
    .withColumn("amount", F.col("debit_amount") - F.col("credit_amount"))

    # Add derived columns (IMPORTANT for KPIs)
    .withColumn("year", F.year("posting_date"))
    .withColumn("month", F.month("posting_date"))
    .withColumn("year_month", F.date_format("posting_date", "yyyy-MM"))

    # Clean data
    .dropDuplicates(["transaction_id"])
    .filter(F.col("transaction_id").isNotNull())
    .filter(F.col("posting_date").isNotNull())

)

# Write to Silver
(df_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("general_ledgers")
)

print("Silver general_ledgers created")

#to display the data from silver general_ledgers
display(spark.read.table("plworkforce_catalog.002_silver.general_ledgers"))